# Đề tài 2: Phân Đoạn Ảnh Y Tế (Kvasir-SEG)
## Google Colab Runner cho Thành viên 2 (SegFormer Transformer Pipeline)

Notebook này được tối ưu riêng cho **Thành viên 2 (TV2 - Transformer)** để huấn luyện và đánh giá **SegFormer (MiT-B0 / MiT-B2)** trên Google Colab GPU (T4).

### Ô 1: Kiểm Tra GPU & Đăng Nhập Weights & Biases (WandB)

In [ ]:
# 1. Kiểm tra phần cứng GPU T4
!nvidia-smi

# 2. Đăng nhập Weights & Biases
import os
import wandb

try:
    from google.colab import userdata
    os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
    wandb.login()
    print("-> Đã đăng nhập WandB thành công từ Colab Secrets!")
except Exception:
    print("-> Nhập API key thủ công từ https://wandb.ai/authorize:")
    wandb.login()

### Ô 2: Clone Mã Nguồn Nhánh `Duc_learnDL` & Cài Đặt Thư Viện

In [ ]:
import os, subprocess

if not os.path.exists("configs/base.yaml"):
    !git clone -b Duc_learnDL https://github.com/daoviet2/DL-Group7.git
    %cd DL-Group7
else:
    !git pull origin Duc_learnDL

# Cài đặt thư viện phụ thuộc
!pip install -r requirements.txt -q

# Tải nhanh bộ dữ liệu Kvasir-SEG (khoảng 10-15 giây)
!python scripts/download_data.py

### Ô 3: Chạy Thử 2 Epochs (Sanity Check) Kiểm Tra Toàn Bộ Pipeline

In [ ]:
# Kiểm tra xem mô hình forward, tính loss, backward và ghi log có hoạt động tốt không
!python scripts/train.py \
  --config configs/segformer_b0_bce_dice.yaml \
  --seed 42 \
  --override train.epochs=2 logging.wandb_mode=offline

### Ô 4: Huấn Luyện Đầy Đủ SegFormer-B0 (50 Epochs, 3 Seeds: 42, 100, 2026)

In [ ]:
# Seed 42
!python scripts/train.py \
  --config configs/segformer_b0_bce_dice.yaml \
  --seed 42 \
  --override logging.wandb_mode=online

# Seed 100
!python scripts/train.py \
  --config configs/segformer_b0_bce_dice.yaml \
  --seed 100 \
  --override logging.wandb_mode=online

# Seed 2026
!python scripts/train.py \
  --config configs/segformer_b0_bce_dice.yaml \
  --seed 2026 \
  --override logging.wandb_mode=online

### Ô 5: Huấn Luyện SegFormer-B2 (50 Epochs)

In [ ]:
!python scripts/train.py \
  --config configs/segformer_b2_bce_dice.yaml \
  --seed 42 \
  --override logging.wandb_mode=online

### Ô 6: Đo Lường Chi Phí Tính Toán (Benchmark Parameters, Latency, FPS, VRAM)

In [ ]:
!python scripts/benchmark_efficiency.py
import pandas as pd
df_bench = pd.read_csv("outputs/efficiency_benchmark.csv")
display(df_bench)

### Ô 7: Đánh Giá Validation & Phân Tích Lỗi Theo Kích Thước Polyp

In [ ]:
!python scripts/evaluate.py \
  --checkpoint outputs/checkpoints/segformer_b0_bce_dice_s42_256/best.pth \
  --split val

### Ô 8: Sao Lưu Bảng Kết Quả Về Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!cp outputs/results.csv /content/drive/MyDrive/segformer_results.csv
print("-> Đã sao lưu results.csv vào Google Drive cá nhân thành công!")